# Direct Preference Optimization (DPO) for YouTube Title Generation
Implement Direct Preference Optimization (DPO) to fine-tune a Large Language Model for generating engaging YouTube titles. DPO is a method for training language models from preference data without the need for an explicit reward model or reinforcement learning.

Use the Qwen3-14B model and fine-tune it using preference pairs from a YouTube titles dataset, comparing chosen vs. rejected title examples.

## Objectives
- Understand the DPO framework and its advantages over RLHF
- Implement DPO fine-tuning using the TRL (Transformers Reinforcement Learning) library
- Work with preference datasets containing chosen/rejected pairs
- Use efficient fine-tuning techniques with LoRA (Low-Rank Adaptation)
- Evaluate model performance before and after DPO training
- Apply modern optimization libraries like Unsloth for accelerated training

## Required Libraries and Setup
- **Core dependencies:** transformers, datasets, trl, torch
- **Efficient training:** unsloth, peft, bitsandbytes
- **Utilities:** accelerate, pandas
- **GPU requirements:** CUDA-compatible GPU with sufficient VRAM

## 3.4.1 Environment Setup and Model Loading

### Step 1: Package Installation and Configuration

In [ ]:
!pip install datasets torch bitsandbytes xformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 41.1 MB/s eta 0:00:00


In [ ]:
!pip install unsloth trl accelerate  --upgrade
!pip install llm-blender
!pip install transformers==4.51.3 --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 10.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Succe

In [ ]:
!pip install trl mergekit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
INFO: pip is still looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.9/104.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.7/431.7 kB 27.9 MB/s

In [ ]:
!pip show unsloth trl transformers | grep -E "Name|Version"

Name: unsloth
Version: 2026.4.6
Name: trl
Version: 0.24.0
Name: transformers
Version: 4.57.6


### Step 2: Dataset Loading and Exploration

In [ ]:
# Authenticate with Hugging Face (if using gated models)
from google.colab import userdata
from huggingface_hub import login
import os

hf_token = userdata.get('HF_TOKEN')

os.environ["HF_TOKEN"] = hf_token

login(hf_token)

print("✅ Hugging Face login in successfully！")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Hugging Face login in successfully！


In [ ]:
# Load the YouTube titles DPO dataset
from datasets import load_dataset, DatasetDict
import json

dataset = load_dataset("EliasHossain/youtube-titles-dpo")

split = dataset["train"].train_test_split(test_size=0.1, seed=42)

dataset = DatasetDict({
    "train": split["train"],
    "valid": split["test"]
})

# Explore the dataset structure
print("\n --- Column Names ---")
print(dataset["train"].column_names)

print("\n --- Sample Data ---")
print(json.dumps(dataset["train"][0], indent=2, ensure_ascii=False))
# - Understand the "prompt", "chosen", and "rejected" fields
# - Check dataset sizes for train and validation splits

print(dataset)


README.md:   0%|          | 0.00/822 [00:00<?, ?B/s]

data/train-00000-of-00001-76f6a471166630(…):   0%|          | 0.00/39.2k [00:00<?, ?B/s]

data/valid-00000-of-00001-a3be3f52cf9748(…):   0%|          | 0.00/14.0k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1026 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/114 [00:00<?, ? examples/s]


 --- Column Names ---
['prompt', 'chosen', 'rejected']

 --- Sample Data ---
{
  "prompt": [
    {
      "content": "Given the YouTube video idea write an engaging title.\n\n**Video Idea**: power law scaling on youtube\n\n**Additional Guidance**:\n- Title should be between 30 and 75 characters long\n- Only return the title idea, nothing else!",
      "role": "user"
    }
  ],
  "chosen": [
    {
      "content": "Power Law Scaling: The Key to YouTube Success",
      "role": "assistant"
    }
  ],
  "rejected": [
    {
      "content": "From Zero to Hero: Scaling YouTube with Power Laws",
      "role": "assistant"
    }
  ]
}
DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 923
    })
    valid: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 103
    })
})


### Step 3: Model and Tokenizer Setup

We are gonna use the unsloth library which is faster at fine-tuning job. Before that we need to connect with T4 GPU from the runtime menu.

In [ ]:
# Make sure you have GPU access
!nvidia-smi

Wed Apr 22 00:03:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Install the required library to use the unsloth

In [ ]:
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# We have to check which Torch version for Xformers (2.3 -> 0.0.27)
from torch import __version__; from packaging.version import Version as V
# Correctly determine and install xformers based on the installed torch version
xformers_version = "xformers==0.0.27" if V(__version__) < V("2.4.0") else "xformers>=0.0.30"
!pip install --no-deps {xformers_version} trl peft accelerate bitsandbytes triton

In [ ]:
import unsloth
from unsloth import FastLanguageModel
from trl import SFTTrainer, DPOTrainer
from trl import DPOConfig
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from datasets import load_dataset, Dataset
import pandas as pd
import json, yaml
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
!pip install -U bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

# Set the model name
model_name = "unsloth/Qwen3-14B-unsloth-bnb-4bit"

# Load model and tokenizer with Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)


# Set pad_token
tokenizer.pad_token = tokenizer.eos_token

==((====))==  Unsloth 2026.4.6: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.59G [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/Qwen3-14B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


## 3.4.2 LoRA Configuration and Base Model Testing

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

### LoRA Adapter Configuration

In [ ]:
# Configure LoRA parameters
model = FastLanguageModel.get_peft_model(
     model,
     r = 32,
     target_modules = [
         "q_proj", "k_proj", "v_proj", "o_proj",
         "gate_proj", "up_proj", "down_proj"
     ],
     lora_alpha = 32,
     lora_dropout = 0,
     bias = "none",
     use_gradient_checkpointing = "unsloth",
     random_state = 3407,
     use_rslora = False,
     loftq_config = None,
 )

# Document the configuration choices and parameter reduction achieved
model.print_trainable_parameters()

Unsloth 2026.4.6 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


trainable params: 128,450,560 || all params: 14,896,757,760 || trainable%: 0.8623


### Base Model Evaluation

Generate titles with base model before fine-tuning:

In [ ]:
from transformers import pipeline
FastLanguageModel.for_inference(model)

# Create a function to format chat prompts properly
def format_chat_prompt(user_input, system_message="You are a helpful assistant."):
     user_prompt = f"<|im_start|>user\\n{user_input}<|im_end|>\\n"
     assistant_prompt = "<|im_start|>assistant\\n"
     formatted_prompt = user_prompt + assistant_prompt
     return formatted_prompt

# Set up text generation pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer)

# Get sample prompts from the validation dataset
prompt = format_chat_prompt(dataset['valid']['prompt'][0][0]['content'])

# Generate output with the base model
outputs = generator(
    prompt,
    max_length=100,
    truncation=True,
    num_return_sequences=1,
    temperature=0.7,
    return_full_text=False)

# Print and analyze the baseline performance
print(outputs[0]['generated_text'])

Device set to use cuda:0


<think>
Okay, the user wants a YouTube video title about multimodal AI. Let me start by understanding what multimodal AI is. It's AI that can process multiple types of data, like text, images, audio


**Analysis:**

The model does not generate a valid YouTube title yet. The output is still in its internal reasoning phase.


In [ ]:
import re, random
FastLanguageModel.for_inference(model)

def format_chat_prompt(user_input, system_message="You are a helpful assistant."):
    return (
        f"<|im_start|>system\n{system_message}<|im_end|>\n"
        f"<|im_start|>user\n{user_input}<|im_end|>\n"
        f"<|im_start|>assistant\n<think>\n\n</think>\n"
    )

def clean_output(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    return text.replace("<|im_end|>", "").strip()

def generate(prompt_text, use_adapter=True):
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]


    if use_adapter:
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    else:
        with model.disable_adapter():
            output = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

    generated = tokenizer.decode(output[0][input_len:], skip_special_tokens=True)
    return clean_output(generated)


total_valid_samples = len(dataset['valid'])
sample_indices = random.sample(range(total_valid_samples), min(5, total_valid_samples))
base_results = {}

for idx, data_idx in enumerate(sample_indices):
    item = dataset['valid'][data_idx]
    raw_prompt = item['prompt']
    user_input_text = raw_prompt[0]['content'] if isinstance(raw_prompt, list) else raw_prompt
    prompt = format_chat_prompt(user_input_text)
    base_results[data_idx] = {
        "user_input": user_input_text,
        "base_output": generate(prompt, use_adapter = False)
    }
    print(f"Example {idx+1}: {base_results[data_idx]['base_output']}")

FastLanguageModel.for_inference(model)
for idx, (data_idx, result) in enumerate(base_results.items()):
    prompt = format_chat_prompt(result["user_input"])
    dpo_output = generate(prompt, use_adapter = True)
    print(f"===== Example {idx+1} =====")
    print(f"📝 Prompt: {result['user_input']}")
    print(f"🐢 Base:   {result['base_output']}")
    print(f"🚀 DPO:    {dpo_output}")

Example 1: "Master Advanced Reasoning: Fine-Tuning LLMs to Think Better"
Example 2: "Build Data Pipelines with Python & GitHub Actions"
Example 3: Build an AI App in 4 Days: A Step-by-Step Guide
Example 4: "LLMs Explained: How AI Models Work & Their Real-World Impact"
Example 5: From Physics to Data Science: My Journey to Becoming a Data Scientist
===== Example 1 =====
📝 Prompt: Given the YouTube video idea write an engaging title.

**Video Idea**: fine-tuning llms to think (advanced reasoning)

**Additional Guidance**:
- Title should be between 30 and 75 characters long
- Only return the title idea, nothing else!
🐢 Base:   "Master Advanced Reasoning: Fine-Tuning LLMs to Think Better"
🚀 DPO:    "Master Advanced Reasoning: Fine-Tuning LLMs to Think Better"
===== Example 2 =====
📝 Prompt: Given the YouTube video idea write an engaging title.

**Video Idea**: building data pipelines with python and github actions

**Additional Guidance**:
- Title should be between 30 and 75 characters lon

## 3.4.3 DPO Training Implementation

### DPO Training Configuration

In [ ]:
from trl import DPOTrainer, DPOConfig
from transformers import TrainingArguments
import pandas as pd

# Define DPO training configuration with DPOConfig
# Configure:
# - output_dir: directory to save checkpoints
# - per_device_train_batch_size: batch size for training (e.g., 4-8)
# - per_device_eval_batch_size: batch size for evaluation
# - num_train_epochs: number of epochs (e.g., 3)
# - learning_rate: learning rate (e.g., 5e-5)
# - lr_scheduler_type: learning rate scheduler (e.g., "linear")
# - warmup_ratio: warmup ratio (e.g., 0.1)
# - logging_steps: logging frequency
# - save_strategy: save strategy (e.g., "epoch")
# - eval_strategy: evaluation strategy (e.g., "epoch")
# - load_best_model_at_end: whether to load best model
# - metric_for_best_model: metric for best model
# - seed: random seed (42)
training_args = DPOConfig(
    output_dir="./dpo_youtube_checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    learning_rate=5e-6,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="steps",
    save_steps = 100,
    eval_strategy="steps",
    eval_steps = 100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    seed=42,
    optim="adamw_8bit",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
)

# Initialize DPOTrainer
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    beta=0.1,
    train_dataset=dataset["train"],
    eval_dataset=dataset["valid"],
    tokenizer=tokenizer,
 )

# Start training
trainer.train()

# After training, log and report the following DPO-specific metrics from trainer.state.log_history:
# - rewards/chosen       (reward margin for preferred responses)
# - rewards/rejected     (reward margin for rejected responses)
# - rewards/accuracies   (how often chosen > rejected)
# - rewards/margins      (difference between chosen and rejected rewards)
# - train/loss and eval/loss
log_history = trainer.state.log_history
train_logs = []
eval_logs = []

for log in log_history:
    if "loss" in log and "eval_loss" not in log:
        train_logs.append({
            "Step": log.get("step"),
            "Epoch": log.get("epoch"),
            "Train Loss": log.get("loss"),
            "Chosen Reward": log.get("rewards/chosen"),
            "Rejected Reward": log.get("rewards/rejected"),
            "Reward Margin": log.get("rewards/margins"),
            "Accuracy": log.get("rewards/accuracies")
        })
    elif "eval_loss" in log:
        eval_logs.append({
            "Step": log.get("step"),
            "Epoch": log.get("epoch"),
            "Eval Loss": log.get("eval_loss"),
            "Eval Margin": log.get("eval_rewards/margins"),
            "Eval Accuracy": log.get("eval_rewards/accuracies")
        })

train_df = pd.DataFrame(train_logs)
eval_df = pd.DataFrame(eval_logs)

print("\n" + "="*20 + " Evaluation " + "="*20)
if not eval_df.empty:
    print(eval_df.to_string(index=False))
else:
    print("Do not have eval data yet.")

print("\n" + "="*20 + " 📈 Training " + "="*20)
if not train_df.empty:
    print(train_df.tail(10).to_string(index=False))

Extracting prompt in train dataset (num_proc=6):   0%|          | 0/923 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/923 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/923 [00:00<?, ? examples/s]

Extracting prompt in eval dataset (num_proc=6):   0%|          | 0/103 [00:00<?, ? examples/s]

Applying chat template to eval dataset (num_proc=6):   0%|          | 0/103 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=6):   0%|          | 0/103 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 923 | Num Epochs = 3 | Total steps = 348
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 128,450,560 of 14,896,757,760 (0.86% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
100,0.626300,0.615652,0.049606,-0.198535,0.711538,0.248140,-80.524849,-89.357857,-2.530631,-2.492466
200,0.536100,0.572205,0.272715,-0.164368,0.711538,0.437083,-78.293762,-89.016197,-2.562839,-2.535015
300,0.484500,0.558308,0.264841,-0.272671,0.692308,0.537513,-78.372498,-90.099236,-2.574246,-2.552008



==================== Evaluation ====================
 Step    Epoch  Eval Loss  Eval Margin  Eval Accuracy
  100 0.865801   0.615652     0.248140       0.711538
  200 1.727273   0.572205     0.437083       0.711538
  300 2.588745   0.558308     0.537513       0.692308

==================== 📈 Training ====================
 Step    Epoch  Train Loss  Chosen Reward  Rejected Reward  Reward Margin  Accuracy
  250 2.155844      0.4990       0.262098        -0.366116       0.628214    0.7750
  260 2.242424      0.5493       0.281935        -0.270485       0.552420    0.7375
  270 2.329004      0.5017       0.362508        -0.235292       0.597800    0.7875
  280 2.415584      0.4425       0.361793        -0.424948       0.786741    0.8500
  290 2.502165      0.4692       0.355104        -0.373392       0.728496    0.7875
  300 2.588745      0.4845       0.280265        -0.449360       0.729625    0.7250
  310 2.675325      0.4621       0.295272        -0.462480       0.757753    0.8375
  32

In [ ]:
print("\n" + "="*20 + " 📈 Training " + "="*20)
if not train_df.empty:
    print(train_df.tail(20).to_string(index=False))


==================== 📈 Training ====================
 Step    Epoch  Train Loss  Chosen Reward  Rejected Reward  Reward Margin  Accuracy
  150 1.294372      0.5852       0.311292        -0.034815       0.346107    0.7125
  160 1.380952      0.6064       0.237484        -0.030609       0.268093    0.6750
  170 1.467532      0.5384       0.247863        -0.185827       0.433690    0.7250
  180 1.554113      0.5295       0.218098        -0.259557       0.477655    0.7750
  190 1.640693      0.4965       0.298017        -0.262284       0.560301    0.8250
  200 1.727273      0.5361       0.210804        -0.292598       0.503402    0.7500
  210 1.813853      0.5022       0.419467        -0.248139       0.667606    0.7750
  220 1.900433      0.4959       0.325066        -0.277385       0.602451    0.7250
  230 1.987013      0.4887       0.296332        -0.364211       0.660543    0.8000
  240 2.069264      0.5408       0.318442        -0.197024       0.515466    0.7500
  250 2.155844      0.

### DPO Trainer Setup and Execution

## 3.5 Model Evaluation and Comparison

Load and evaluate the fine-tuned model:

In [ ]:
import random
FastLanguageModel.for_inference(trainer.model)

def generate_dpo(prompt_text):
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]
    output = trainer.model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    return clean_output(tokenizer.decode(output[0][input_len:], skip_special_tokens=True))


print("=" * 60)
for idx, (data_idx, result) in enumerate(base_results.items()):
    item = dataset['valid'][data_idx]
    raw_chosen = item.get('chosen', '')
    chosen_text = raw_chosen[-1]['content'] if isinstance(raw_chosen, list) else raw_chosen

    prompt = format_chat_prompt(result["user_input"])
    dpo_output = generate_dpo(prompt)

    print(f"===== Example {idx+1} (Index: {data_idx}) =====")
    print(f"📝 Prompt:  {result['user_input']}")
    print(f"🎯 Chosen:  {chosen_text}")
    print(f"🐢 Base:    {result['base_output']}")
    print(f"🚀 DPO:     {dpo_output}")
    print("=" * 60 + "\n")

===== Example 1 (Index: 38) =====
📝 Prompt:  Given the YouTube video idea write an engaging title.

**Video Idea**: fine-tuning llms to think (advanced reasoning)

**Additional Guidance**:
- Title should be between 30 and 75 characters long
- Only return the title idea, nothing else!
🎯 Chosen:  How to Make LLMs Think Better: Tips Revealed
🐢 Base:    "Master Advanced Reasoning: Fine-Tuning LLMs to Think Better"
🚀 DPO:     Fine-Tuning LLMs for Advanced Reasoning: A Deep Dive

===== Example 2 (Index: 24) =====
📝 Prompt:  Given the YouTube video idea write an engaging title.

**Video Idea**: building data pipelines with python and github actions

**Additional Guidance**:
- Title should be between 30 and 75 characters long
- Only return the title idea, nothing else!
🎯 Chosen:  Automate Data Pipelines Using Python & GitHub Actions
🐢 Base:    "Build Data Pipelines with Python & GitHub Actions"
🚀 DPO:     Build Data Pipelines with Python and GitHub Actions

===== Example 3 (Index: 40) =====
📝 

# Compare base model vs fine-tuned model outputs

- Training results were stable and healthy throughout all 3 epochs. Train Loss decreased from 0.626 to 0.485, while Validation Loss dropped from 0.616 to 0.558, with a consistently small gap between the two, indicating no overfitting. Most importantly, Reward Margin more than doubled from 0.248 to 0.537, confirming that the model successfully learned meaningful preference signals from the DPO training data.

Before the final run:
- Where DPO improved: Titles are more specific and audience-aware.
Stronger verb-first openings create better engagement hooks
Reward margin growth (0.25 → 0.54) confirms the model learned meaningful preference signals

- Where DPO did not improve: Character limit compliance was not consistently better than Base. Qwen3's thinking mode caused output failures on 2 out of 5 examples for both models. Example 5 showed DPO over-generating content beyond what the task required.

In the final run:
- Disabling thinking mode resolved all formatting issues seen in the previous run. Both models now produce clean, compliant outputs across all 5 examples.

# Generate multiple examples and document the improvements
**Example1 - fine-tuning LLMs to think**

Both outputs comply with the character limit. The Base title relies on the overused "Master" opener. The DPO title is more structured and the phrase "A Deep Dive" signals content depth, which is more appealing to a technical audience. DPO wins. Neither model outperforms Chosen.

**Example2 — building data pipelines with Python and GitHub Actions**

The two outputs are nearly identical. The only difference is & vs and. Neither model captured the stronger action verb "Automate" used in the Chosen title, which conveys a clearer value proposition.

**Example3 — building an AI app in 4 days**

Both models produced identical outputs, meaning DPO training had no differentiating effect on this prompt. The Chosen title uses first-person narrative ("How I Built"), which creates a more personal and story-driven hook, which is a style neither model learned to replicate.

**Example4 — LLMs**
This is the most interesting example. The Base model takes an explanatory approach while DPO takes a forward-looking, big-picture angle. The Chosen title uses a competitive framing ("vs. Humans") and a time anchor ("2025") to create curiosity and urgency. The DPO title is closer in spirit to this "grand narrative" style. DPO wins.


**Example5 — from physics to data science**

Again identical outputs. Both models generated a longer, more formal title, while the Chosen title is concise, conversational, and better suited to a personal vlog format. This suggests the model has not sufficiently learned the preference for brevity and casual tone in personal story content.

# Conclusion

DPO fine-tuning produced observable but limited improvements. On Examples 1 and 4, the DPO model generated more distinctive and engaging titles with a clearer stylistic direction. However, on Examples 2, 3, and 5, the outputs were identical to the Base model, suggesting the model has not yet developed a strong or consistent preference bias across all prompt types.

Two factors explain this limitation. First, the training dataset contains noisy or stylistically inconsistent chosen labels, making it harder for the model to learn a unified preference direction. Second, 3 epochs of training may not be sufficient — a Reward Margin of 0.54, while improved, still has room to grow. Cleaning the dataset and extending training to 5+ epochs are the recommended next steps to further improve output quality.